# Coding Agent

You'll need to build a Coding Agent powered by an LLM that can:
- Clone and explore GitHub repositories
- Read, analyze, and modify code files
- Execute tasks autonomously based on natural language instructions

In [19]:
!pip install openai chromadb "langfuse<3" pyyaml tiktoken requests -q #en las versiones posteriores a 3 no soporta el .trace()

#### Create OpenAI Client

In [20]:
import os
import json
import subprocess
import requests, yaml, re, hashlib
from pathlib import Path
from openai import OpenAI
from datetime import datetime

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    langfuse_pk = userdata.get("LANGFUSE_PK")
    langfuse_sk = userdata.get("LANGFUSE_SK")
except Exception:
    api_key = input("Enter your API key: ")
    langfuse_pk = input("Langfuse public key: ")
    langfuse_sk = input("Langfuse secret key: ")

client = OpenAI(
    api_key=api_key
)

MODEL = "gpt-5-nano"
EMBED_MODEL = "text-embedding-3-small"

#creo un directorio de trbajo donde operará el agente
WORKSPACE = Path("/content/workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)

print("Cliente OpenAI listo")
print(f"Worksapce: {WORKSPACE}")



Cliente OpenAI listo
Worksapce: /content/workspace


In [21]:
from langfuse import Langfuse

lf = Langfuse(
    public_key=langfuse_pk,
    secret_key=langfuse_sk,
    host="https://cloud.langfuse.com"
)

try:
    lf.auth_check()
    print("Langfuse conectado")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")
    print(f"public_key empieza con: {langfuse_pk[:8] if langfuse_pk else 'VACIA'}")
    print(f"secret_key empieza con: {langfuse_sk[:8] if langfuse_sk else 'VACIA'}")

Langfuse conectado


In [22]:
config_yaml = """
workspace: /content/workspace

permissions:
  read:
    deny:
      - ".env"
      - "**/*.pem"
      - "**/*.key"
      - "secrets/**"
  write:
    deny:
      - ".github/**"
      - "package-lock.json"
      - "yarn.lock"
commands:
  deny:
    - "rm -rf"
    - "git push"
    - "sudo"
    - "chmod"
    - "curl | bash"
    - "wget | bash"
  require_approval:
    - "npm install"
    - "pip install"
    - "git commit"
"""

config_path = WORKSPACE / "agent.config.yaml"
config_path.write_text(config_yaml.strip())
print(f"agent.config.yaml creado en {config_path}")

agent.config.yaml creado en /content/workspace/agent.config.yaml


In [23]:
import fnmatch # ofrece coincidencia de patrones al estilo de los comodines de terminal Unix

def load_config(path=None) -> dict:
    path = path or WORKSPACE / "agent.config.yaml"
    with open(path, "r") as f:
        return yaml.safe_load(f)

CONFIG = load_config()

def matches_any(value: str, patterns: list) -> bool: #¿existe match entre value y algún pattern de la lista?
    for pattern in patterns:
        if fnmatch.fnmatch(value, pattern):
            return True
    return False

def validate_tool_call(tool_name: str, args: dict) -> str | None:

    # guardrails originales
    if tool_name in ("read_file", "write_file", "list_files"):
        path = args.get("path") or args.get("directory", "..")
        abs_path = str(Path(path).resolve())
        for blocked in GUARDRAILS.get("blocked_paths", []):
            if abs_path.startswith(str(Path(blocked).resolve())):
                return f"Acceso bloqueado a '{path}' por guardrails."
        allowed = GUARDRAILS.get("allowed_directories", [])
        if allowed:
            if not any(abs_path.startswith(str(Path(d).resolve())) for d in allowed):
                return f"'{path}' está fuera de los directorios permitidos."

    # políticas del YAML
    perms = CONFIG.get("permissions", {})
    cmds  = CONFIG.get("commands", {})

    if tool_name == "read_file":
        path = args.get("path", "")
        if matches_any(path, perms.get("read", {}).get("deny", [])):
            return f"[BLOQUEADO] read_file: '{path}' denegado por config."

    if tool_name == "write_file":
        path = args.get("path", "")
        if matches_any(path, perms.get("write", {}).get("deny", [])):
            return f"[BLOQUEADO] write_file: '{path}' denegado por config."

    if tool_name == "run_command":
        command = args.get("command", "")
        for blocked in cmds.get("deny", []):
            if blocked in command:
                return f"[BLOQUEADO] run_command: '{blocked}' es un comando prohibido."
        for needs_ok in cmds.get("require_approval", []):
            if needs_ok in command:
                print(f"\n[APROBACIÓN REQUERIDA] El agente quiere ejecutar: {command}")
                choice = input("¿Permitir? (s/n): ").strip().lower()
                if choice != "s":
                    return f"[CANCELADO] El usuario rechazó ejecutar: {command}"

    return None

print("Configuración cargada:")
print(f"  - read deny:  {CONFIG['permissions']['read']['deny']}")
print(f"  - write deny: {CONFIG['permissions']['write']['deny']}")
print(f"  - cmd deny:   {CONFIG['commands']['deny']}")
print(f"  - approval:   {CONFIG['commands']['require_approval']}")

Configuración cargada:
  - read deny:  ['.env', '**/*.pem', '**/*.key', 'secrets/**']
  - write deny: ['.github/**', 'package-lock.json', 'yarn.lock']
  - cmd deny:   ['rm -rf', 'git push', 'sudo', 'chmod', 'curl | bash', 'wget | bash']
  - approval:   ['npm install', 'pip install', 'git commit']


In [24]:
#indexación de documentación React
REACT_DOCS_SOURCES = [
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/index.md",
        "title": "React - Learn"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/thinking-in-react.md",
        "title": "Thinking in React"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/passing-props-to-a-component.md",
        "title": "Props"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/managing-state.md",
        "title": "Managing State"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/reference/react/hooks.md",
        "title": "Hooks Reference"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/scaling-up-with-reducer-and-context.md",
        "title": "Reducer and Context"
    },
]

def fetch_doc(url: str) -> str:
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  [WARN] No se pudo descargar {url}: {e}")
        return ""

raw_docs = []
for source in REACT_DOCS_SOURCES:
    print(f"Descargando: {source['title']}...")
    content = fetch_doc(source["url"])
    if content:
        raw_docs.append({"title": source["title"], "url": source["url"], "content": content})
        print(f"{len(content)} chars")

print(f"{len(raw_docs)} documentos descargados")

Descargando: React - Learn...
15777 chars
Descargando: Thinking in React...
22965 chars
Descargando: Props...
26905 chars
Descargando: Managing State...
25302 chars
Descargando: Hooks Reference...
5665 chars
Descargando: Reducer and Context...
30942 chars
6 documentos descargados


In [25]:
def chunk_text(text: str, title: str, url: str, chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """Divide el texto en chunks por palabras con overlap."""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        chunk_text = " ".join(chunk_words)
        chunk_id = hashlib.md5(f"{url}:{i}".encode()).hexdigest()
        chunks.append({
            "id": chunk_id,
            "text": chunk_text,
            "title": title,
            "url": url,
            "chunk_index": len(chunks)
        })
        i += chunk_size - overlap
    return chunks

def get_embedding(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text[:8000]  # límite de seguridad
    )
    return response.data[0].embedding

# Generar todos los chunks
all_chunks = []
for doc in raw_docs:
    chunks = chunk_text(doc["content"], doc["title"], doc["url"])
    all_chunks.extend(chunks)
    print(f"  {doc['title']}: {len(chunks)} chunks")

print(f"\n✓ Total chunks: {len(all_chunks)}")

  React - Learn: 5 chunks
  Thinking in React: 7 chunks
  Props: 8 chunks
  Managing State: 7 chunks
  Hooks Reference: 2 chunks
  Reducer and Context: 8 chunks

✓ Total chunks: 37


In [26]:
import numpy as np

# Guardar chunks y embeddings en memoria (listas paralelas)
rag_texts     = []
rag_embeddings = []
rag_metadata  = []

print("Indexando chunks...")
for i, chunk in enumerate(all_chunks):
    embedding = get_embedding(chunk["text"])
    rag_texts.append(chunk["text"])
    rag_embeddings.append(embedding)
    rag_metadata.append({"title": chunk["title"], "url": chunk["url"]})
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(all_chunks)} chunks indexados...")

# Convertir a matriz numpy para búsqueda eficiente
rag_matrix = np.array(rag_embeddings)  # shape: (n_chunks, 1536)
print(f"\n✓ RAG listo — {len(rag_texts)} chunks indexados")

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Similitud coseno entre vector a y matriz b."""
    a_norm = a / np.linalg.norm(a)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return b_norm @ a_norm

def rag_search(query: str, n_results: int = 3) -> list[dict]:
    """Busca en el RAG por similitud coseno. Retorna chunks con fuente etiquetada."""
    query_embedding = np.array(get_embedding(query))
    scores = cosine_similarity(query_embedding, rag_matrix)
    top_indices = np.argsort(scores)[::-1][:n_results]

    results = []
    for idx in top_indices:
        results.append({
            "text": rag_texts[idx],
            "title": rag_metadata[idx]["title"],
            "url": rag_metadata[idx]["url"],
            "score": float(scores[idx]),
            "source": "RAG"
        })
    return results

# Test rápido
print("\nTest RAG:")
resultados = rag_search("React hooks useState useEffect")
for r in resultados:
    print(f"  [{r['source']}] {r['title']} (score: {r['score']:.3f})")
    print(f"    {r['text'][:100]}...")

Indexando chunks...
  10/37 chunks indexados...
  20/37 chunks indexados...
  30/37 chunks indexados...

✓ RAG listo — 37 chunks indexados

Test RAG:
  [RAG] Props (score: 0.532)
    import { useState, useEffect } from 'react'; import Clock from './Clock.js'; function useTime() { co...
  [RAG] Hooks Reference (score: 0.527)
    a non-reactive event to fire from any Effect hook. --- ## Performance Hooks {/*performance-hooks*/} ...
  [RAG] Managing State (score: 0.524)
    ); } let nextId = 3; ``` ```js src/TaskList.js import { useState, useContext } from 'react'; import ...


In [27]:
# ── task_state: estado compartido entre subagentes (vive en memoria por sesión) ──
def new_task_state(request: str, repo_path: str) -> dict:
    return {
        "original_request": request,
        "repo_path": repo_path,
        "progress": [],
        "subagent_results": {
            "explorer": None,
            "researcher": None,
            "implementer": None,
            "tester": None,
            "reviewer": None
        },
        "sources_consulted": [],   # qué docs del RAG o web se usaron
        "files_modified": [],
        "observations": [],
        "rag_chunks_used": []      # fragmentos recuperados del RAG
    }

def log_progress(state: dict, subagent: str, message: str):
    entry = f"[{subagent.upper()}] {message}"
    state["progress"].append(entry)
    print(entry)

# ── project_memory: persiste entre sesiones en disco ──
MEMORY_PATH = WORKSPACE / "project_memory.json"

def load_memory() -> dict:
    if MEMORY_PATH.exists():
        with open(MEMORY_PATH) as f:
            memory = json.load(f)
        print(f"✓ Memoria cargada ({len(memory.get('sessions', []))} sesiones previas)")
        return memory
    return {
        "sessions": [],
        "architecture": {},
        "key_files": [],
        "dependencies": {},
        "conventions": [],
        "decisions": [],
        "bugs_investigated": []
    }

def save_memory(memory: dict):
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f, indent=2, ensure_ascii=False)

def update_memory(memory: dict, state: dict):
    """Al final de cada sesión, actualiza la memoria persistente con lo aprendido."""
    session_summary = {
        "date": datetime.now().isoformat(),
        "request": state["original_request"],
        "repo_path": state["repo_path"],
        "files_modified": state["files_modified"],
        "observations": state["observations"],
        "sources": state["sources_consulted"]
    }
    memory["sessions"].append(session_summary)

    # actualizar arquitectura si el explorer encontró algo
    if state["subagent_results"]["explorer"]:
        memory["architecture"][state["repo_path"]] = state["subagent_results"]["explorer"]

    save_memory(memory)
    print(f"✓ Memoria guardada en {MEMORY_PATH}")

PROJECT_MEMORY = load_memory()
print(f"✓ task_state y project_memory listos")

✓ task_state y project_memory listos


In [28]:
import tiktoken

TOKENIZER = tiktoken.encoding_for_model("gpt-4o-mini")
MAX_CONTEXT_TOKENS = 6000  # dejamos margen del límite del modelo

def count_tokens(messages: list) -> int:
    """Cuenta tokens aproximados del historial."""
    total = 0
    for m in messages:
        content = m.get("content") or ""
        total += len(TOKENIZER.encode(str(content)))
    return total

def summarize_history(messages: list) -> list:
    """
    Si el historial es muy largo, resume los mensajes del medio
    y conserva el system prompt + últimos 4 mensajes.
    """
    system = [m for m in messages if m["role"] == "system"]
    rest   = [m for m in messages if m["role"] != "system"]

    if len(rest) <= 4:
        return messages  # no hace falta resumir

    to_summarize = rest[:-4]
    recent       = rest[-4:]

    history_text = "\n".join(
        f"{m['role'].upper()}: {str(m.get('content', ''))[:300]}"
        for m in to_summarize
    )

    summary_response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"Resumí en 3-5 oraciones qué se hizo hasta ahora:\n\n{history_text}"
        }]
    )
    summary = summary_response.choices[0].message.content

    summary_msg = {
        "role": "system",
        "content": f"[RESUMEN DE CONTEXTO PREVIO]\n{summary}"
    }

    print(f"  [CONTEXTO] Historial resumido ({len(to_summarize)} mensajes → 1 resumen)")
    return system + [summary_msg] + recent


def detect_loop(state: dict, tool_name: str, args: dict) -> bool:
    """
    Detecta si el agente está repitiendo la misma acción sin avanzar.
    Retorna True si detecta loop.
    """
    # buscar en el progreso cuántas veces se llamó esta tool con estos mismos args
    key = f"{tool_name}:{json.dumps(args, sort_keys=True)}"
    repetitions = sum(1 for p in state["progress"] if key in p)

    if repetitions >= 2:
        print(f"  [LOOP DETECTADO] '{tool_name}' repetido {repetitions+1} veces sin avance.")
        return True
    return False


def inner_loop_with_guards(messages: list, state: dict, tools_schema: list,
                            supervision: bool, subagent_name: str) -> str:
    """
    Inner loop extendido con detección de loops y manejo de contexto.
    Reemplaza al inner_loop original para los subagentes.
    """
    iteracion = 0

    while True:
        iteracion += 1

        # manejo de contexto largo
        if count_tokens(messages) > MAX_CONTEXT_TOKENS:
            messages = summarize_history(messages)

        print(f"  [{subagent_name.upper()}] iteración {iteracion}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content})
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                }
                for tc in msg.tool_calls
            ]
        })

        for call in msg.tool_calls:
            tool_name = call.function.name
            tool_args = json.loads(call.function.arguments)

            # detección de loop
            if detect_loop(state, tool_name, tool_args):
                loop_msg = (
                    f"Detecté que estoy repitiendo '{tool_name}' sin avanzar. "
                    f"Voy a cambiar de estrategia o detenerme."
                )
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": loop_msg
                })
                log_progress(state, subagent_name, f"LOOP DETECTADO en {tool_name} — cambiando estrategia")
                continue

            log_progress(state, subagent_name, f"tool: {tool_name} {list(tool_args.values())[:1]}")
            result = execute_tool(tool_name, tool_args, supervision)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result) if result is not None else "Error: la tool no devolvió resultado"
            })

print("✓ Guards listos: detección de loops + manejo de contexto")

✓ Guards listos: detección de loops + manejo de contexto


In [29]:
# ── subagentes: cada uno usa inner_loop_with_guards con su propio system prompt ──

def _run_subagent(state: dict, supervision: bool, name: str, objetivo: str, extra_context: str = "") -> str:
    """Arma el mensaje inicial del subagente, corre el loop con guards y guarda el resultado en state."""
    system_prompt = (
        f"Sos el subagente '{name}' dentro de un sistema multi-agente de análisis de código.\n"
        f"Tarea original del usuario: {state['original_request']}\n"
        f"Repositorio: {state['repo_path']}\n\n"
        f"Tu objetivo específico:\n{objetivo}\n"
    )
    if extra_context:
        system_prompt += f"\nContexto adicional:\n{extra_context}\n"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": objetivo}
    ]

    log_progress(state, name, "iniciando")
    resultado = inner_loop_with_guards(messages, state, TOOLS_SCHEMA, supervision, name)
    state["subagent_results"][name] = resultado
    log_progress(state, name, "terminado")
    return resultado


def run_explorer(state: dict, supervision: bool):
    objetivo = (
        "Explorá la estructura del repositorio con list_files y read_file "
        "(package.json, README, carpetas principales de src/). "
        "Identificá: stack tecnológico, estructura de carpetas, convenciones y componentes clave. "
        "No modifiques ningún archivo. Devolvé un resumen conciso de la arquitectura encontrada."
    )
    return _run_subagent(state, supervision, "explorer", objetivo)


def run_researcher(state: dict, supervision: bool):
    query = "React component architecture best practices project structure"
    rag_results = rag_search(query, n_results=3)
    state["rag_chunks_used"].extend(rag_results)
    for r in rag_results:
        state["sources_consulted"].append({"source": "RAG", "title": r["title"], "url": r["url"]})

    rag_context = "\n\n".join(
        f"[{r['title']}] ({r['url']}): {r['text'][:400]}" for r in rag_results
    )

    objetivo = (
        "Investigá buenas prácticas de arquitectura React relevantes al repo analizado. "
        "Usá el contexto de RAG provisto y, si hace falta más información actualizada, "
        "la tool web_search. No modifiques archivos. Devolvé hallazgos clave y recomendaciones."
    )
    resultado = _run_subagent(state, supervision, "researcher", objetivo, extra_context=rag_context)
    state["sources_consulted"].append({"source": "web_search", "title": "researcher queries", "url": ""})
    return resultado


def run_implementer(state: dict, supervision: bool):
    explorer_summary = state["subagent_results"].get("explorer") or ""
    researcher_summary = state["subagent_results"].get("researcher") or ""
    report_path = f"{state['repo_path']}/ARCHITECTURE_REPORT.md"

    objetivo = (
        f"Con base en los hallazgos del explorer y el researcher, redactá un reporte de "
        f"arquitectura completo en Markdown y guardalo con write_file en '{report_path}'. "
        "El reporte debe incluir: stack, estructura de carpetas, patrones usados, "
        "y recomendaciones de mejora."
    )
    extra_context = f"Hallazgos explorer:\n{explorer_summary}\n\nHallazgos researcher:\n{researcher_summary}"
    resultado = _run_subagent(state, supervision, "implementer", objetivo, extra_context=extra_context)

    if Path(report_path).exists() and report_path not in state["files_modified"]:
        state["files_modified"].append(report_path)
    return resultado


def run_tester(state: dict, supervision: bool):
    report_path = f"{state['repo_path']}/ARCHITECTURE_REPORT.md"
    objetivo = (
        f"Verificá con read_file que '{report_path}' exista y tenga contenido completo "
        "(no vacío, con secciones claras). Si encontrás problemas, indicalos. "
        "No modifiques el archivo, solo reportá el resultado de la verificación."
    )
    return _run_subagent(state, supervision, "tester", objetivo)


def run_reviewer(state: dict, supervision: bool):
    explorer_summary = state["subagent_results"].get("explorer") or ""
    researcher_summary = state["subagent_results"].get("researcher") or ""
    implementer_summary = state["subagent_results"].get("implementer") or ""
    tester_summary = state["subagent_results"].get("tester") or ""

    objetivo = (
        "Revisá el trabajo completo del explorer, researcher, implementer y tester. "
        "Dictaminá si el reporte de arquitectura cumple el objetivo original y está listo para entregar. "
        "Empezá tu respuesta con 'APROBADO' o 'RECHAZADO' seguido de una breve justificación."
    )
    extra_context = (
        f"Explorer:\n{explorer_summary}\n\nResearcher:\n{researcher_summary}\n\n"
        f"Implementer:\n{implementer_summary}\n\nTester:\n{tester_summary}"
    )
    return _run_subagent(state, supervision, "reviewer", objetivo, extra_context=extra_context)


print("✓ Subagentes (explorer, researcher, implementer, tester, reviewer) listos")

✓ Subagentes (explorer, researcher, implementer, tester, reviewer) listos


In [30]:
def run_main_agent(repo_url: str = None, repo_path: str = None, supervision: bool = True):
    """
    Agente principal. Recibe un repo (URL o path local),
    coordina los subagentes y genera el reporte de arquitectura.
    """

    # ── 1. clonar repo si se dio URL ──
    if repo_url and not repo_path:
        repo_name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
        repo_path = str(WORKSPACE / repo_name)

        if Path(repo_path).exists():
            print(f"✓ Repo ya clonado en {repo_path} (memoria)")
        else:
            print(f"Clonando {repo_url}...")
            result = subprocess.run(
                ["git", "clone", "--depth", "1", repo_url, repo_path],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                print(f"[ERROR] Clone falló:\n{result.stderr}")
                return
            print(f"✓ Repo clonado en {repo_path}")

    if not repo_path or not Path(repo_path).exists():
        print("[ERROR] Necesitás proveer un repo_url o un repo_path válido.")
        return

    request = f"Analizar el repositorio React en {repo_path} y generar reporte de arquitectura."

    # ── 2. verificar memoria previa ──
    if repo_path in PROJECT_MEMORY.get("architecture", {}):
        print(f"[MEMORIA] Ya analicé este repo antes. Usando contexto previo.")

    # ── 3. inicializar task_state ──
    state = new_task_state(request, repo_path)
    log_progress(state, "main", f"Tarea iniciada: {request}")

    # ── 4. traza Langfuse ──
    # Correcto: Use lf.trace_manager.create_trace() to explicitly create a new trace
    trace = lf.trace(
        name="react-architecture-agent",
        input={"repo_path": repo_path, "request": request},
        metadata={"model": MODEL, "supervision": supervision}
    )

    try:
        # ── 5. ejecutar subagentes en orden ──
        span_explorer = trace.span(name="explorer")
        run_explorer(state, supervision)
        span_explorer.end(output={"result": str(state["subagent_results"]["explorer"])[:500]})

        span_researcher = trace.span(name="researcher")
        run_researcher(state, supervision)
        span_researcher.end(output={
            "result": str(state["subagent_results"]["researcher"])[:500],
            "rag_chunks": len(state["rag_chunks_used"])
        })

        span_implementer = trace.span(name="implementer")
        run_implementer(state, supervision)
        span_implementer.end(output={"files_modified": state["files_modified"]})

        span_tester = trace.span(name="tester")
        run_tester(state, supervision)
        span_tester.end(output={"result": str(state["subagent_results"]["tester"])[:500]})

        span_reviewer = trace.span(name="reviewer")
        run_reviewer(state, supervision)
        span_reviewer.end(output={"result": str(state["subagent_results"]["reviewer"])[:500]})

        # ── 6. actualizar memoria persistente ──
        update_memory(PROJECT_MEMORY, state)

        # ── 7. resumen final ──
        print("\n" + "="*60)
        print("RESUMEN DE EJECUCIÓN")
        print("="*60)
        print(f"Repo:            {repo_path}")
        print(f"Archivos:        {state['files_modified']}")
        print(f"RAG chunks:      {len(state['rag_chunks_used'])}")
        print(f"Fuentes:         {list(set(s['source'] for s in state['sources_consulted']))}")
        print(f"Veredicto:       {state['subagent_results']['reviewer'][:200]}")
        print("\nProgreso:")
        for p in state["progress"]:
            print(f"  {p}")

        trace.update(
            output={"files_modified": state["files_modified"], "status": "completado"},
            metadata={"rag_chunks_used": len(state["rag_chunks_used"])}
        )

    except Exception as e:
        log_progress(state, "main", f"ERROR: {e}")
        trace.update(output={"status": "error", "error": str(e)})
        raise

    finally:
        lf.flush()

    return state

print("✓ Agente principal listo")

✓ Agente principal listo


## Implementación

In [31]:
#implementación de las herramientas

def read_file(path: str, **kwargs) -> str: #--> path --> leo documento
  try:
    with open(path, 'r', encoding='utf-8') as f:
      return f.read()
  except FileNotFoundError:
    return f"Error: el archivo no fue encontrado en '{path}'"
  except Exception as e:
    return f"Error al intentar leer '{path}': {e}"


def write_file(path: str, content:str) -> str: # escribo contenido en un archivo | reemplazo si ya exite
  try:
    Path(path).parent.mkdir(parents= True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
      f.write(content)
      return f"Archivo escrito con éxito en '{path}'"
  except Exception as e:
    return f"Hubo un error al escribir en '{path}': {e}"

def run_command(command: str) -> str: # --> ejecuto comando en terminal --> retorno stdout Y stderr
  try:
    result = subprocess.run(
        command,
        shell=True,
        capture_output = True,
        text=True,
        timeout=45
    )
    output = ""
    if result.stdout:
      output += f"\n\nSTDOUT:\n{result.stdout}"
    if result.stderr:
      output += f"\n\nSTDERR:\n{result.stderr}"
    output += f"\nReturn code: {result.returncode}"
    return output if output.strip() else "No hay output"
  except subprocess.TimeoutExpired:
    return "Error: el comando excedió el timeout de 45 segundos"
  except Exception as e:
    return f"Se produjo un error al ejecutar: {e}"

def list_files(directory:str=".") -> str: # listo un directorio, mínimo para que pueda operar
  try:
    p = Path(directory)
    if not p.exists():
      return "El directorio '{directory} especificado no existe"
    items = sorted(p.iterdir())
    lines = [f"Contenido de '{directory}': "]
    for item in items:
      lines.append(f"{item.name}")
    return "\n".join(lines) if len(lines) > 1 else f"'{directory}' está vacío"
  except Exception as e:
    return f"Error listando '{directory}': {e}"



def web_search(query:str)->str: # herramienta websearch itnegrada de openai
  try:
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": query}],
        tools=[{"type": "web_search_preview"}]

    )
    return response.choices[0].message.content
  except Exception as e:
    return f"Error en web_search: {e}"

In [32]:
# Definición de herramientas

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Lee el contenido completo de un archivo dado su path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path al archivo a leer"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Escribe (o sobreescribe) contenido en un archivo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path del archivo a escribir"},
                    "content": {"type": "string", "description": "Contenido a escribir en el archivo"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Ejecuta un comando de terminal y devuelve stdout y stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "Comando de terminal a ejecutar"}
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "Lista archivos y carpetas en un directorio.",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {"type": "string", "description": "Path del directorio a listar (default: '.')"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Busca información en la web y devuelve los resultados.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Consulta de búsqueda"}
                },
                "required": ["query"]
            }
        }
    }
]

TOOLS_MAP = { # mapeo nombre --> función python
    "read_file": read_file,
    "write_file": write_file,
    "run_command": run_command,
    "list_files": list_files,
    "web_search": web_search,
}

DESTRUCTIVE_TOOLS = {"write_file", "run_command"} # necesita supervisión porque modifican el sistema

# esta es la lista de herramientas que puede ejecutar mi agente. El llm no puede ejecutar código directametne pero si
# pedir ejecutar una función. SIn este esquema, el modelo no puede usar las funciones ya que no sabe cuales existen
# tengo que aclarar cuáles hay, su nombre y los parámetros que speran

### Guardrails

In [33]:
import json

guardrails_config = {
    "allowed_directories": ["/content/workspace"],
    "blocked_paths": ["/etc", "/root"],
    "blocked_commands": ["rm -rf", "git push", "sudo", "chmod"]
}

with open("guardrails.json", "w") as f:
    json.dump(guardrails_config, f, indent=2)

print("guardrails.json creado")

guardrails.json creado


In [34]:
# Funciones para crear archivos restringidos -> el agente no debiera poder acceder o modificarlos

# Archivo con permiso para todos -> el agente no debiera poder hacer chmod
# Intentar chmod 400 test.txt
def create_file_with_full_access(file_name, content):
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    os.chmod(file_name, 0o777)
    print(f"Archivo {file_name} creado con éxito y permisos totales.")

create_file_with_full_access("test.txt", "You shouldn't be able to chmod this!")

# Crear un directorio restringido
# Intentar que acceda
def crear_directorio(nombre_carpeta):
    try:
        # Crea la carpeta.
        # parents=True crea carpetas intermedias si no existen.
        # exist_ok=True evita errores si ya existe.
        os.makedirs(nombre_carpeta, exist_ok=True)
        print(f"Directorio '{nombre_carpeta}' listo.")
    except Exception as e:
        print(f"Error al crear directorio: {e}")

crear_directorio("root")

Archivo test.txt creado con éxito y permisos totales.
Directorio 'root' listo.


In [35]:
def load_guardrails(path="guardrails.json") -> dict:
    try:
        with open(path) as f:
            config = json.load(f)
        print(f"Guardrails cargados: {config}")
        return config
    except FileNotFoundError:
        print("Sin guardrails.json, sin restricciones.")
        return {}

GUARDRAILS = load_guardrails()

Guardrails cargados: {'allowed_directories': ['/content/workspace'], 'blocked_paths': ['/etc', '/root'], 'blocked_commands': ['rm -rf', 'git push', 'sudo', 'chmod']}


### Loops

In [36]:
PROMPT = """Sos un agente de código cuyo trabajo es ayudar al usaurio con sus tareas de código.
Podes usar las herramietnas disponibles, estas son: read_file, write_file, run_command, list_files y web_search.
Respetá los siguietnes pasos al recibir una tarea:
1) Analizá los requisitos y pasos necesarios para resolver el problema
2) Hacé uso de las tools, de forma iterativa, para cumplir los objetivos
3) Verificá que el trabajo hecho sea correcto con, por ejemplo, tests
4) Reportá el resultado al usuario y explicá cómo lo resolviste

Mantené al usuario siempre al tanto de qué y por qué hacés lo que hacés."""


def execute_tool(name: str, args: dict, supervision: bool) -> str: # dict --> []
  args = {k.strip().rstrip('?'): v for k, v in args.items()}

  error = validate_tool_call(name, args) # valido guardrails
  if error:
      print(error)
      return error  # el LLM se entera y busca otra forma

  if supervision and name in DESTRUCTIVE_TOOLS:
    message = name
    if name == "run_command":
      message += " " + " ".join(args.values())
    print(f"\n[SUPERVISIÓN] El agente quiere ejecutar: {message}")
    choice = input("¿Permitir? (s/n): ").strip().lower() # strip elimina por defecto los espacios en blanco, tabulaciones y saltos de línea
    if choice != 's':
      return f"Cancelando {name}..."
  func = TOOLS_MAP[name]
  result = func(**args) #** desempaqueta el diccionario
  return result

def inner_loop(messages: list, supervision:bool)-> str: # es el loop interno. Llama al LLM y ejecuta tools hasta que respodnda sin tool_calls. Return: mensaje final del asistnet
  iteracion = 0
  while True:
    iteracion += 1
    print(f"Loop interno - iteración: {iteracion}")

    response = client.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = TOOLS_SCHEMA,
        tool_choice = "auto"
    )

    msg = response.choices[0].message # primera rta del modelo


    if not msg.tool_calls: # no hay tool calls --> agente terminó turno
        messages.append({"role": "assistant", "content": msg.content})
        return msg.content

    messages.append({
    "role": "assistant",
    "content": msg.content or "",  # convierte null a string vacío
    "tool_calls": [
        {
            "id": tc.id,
            "type": "function",
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments
            }
        }
        for tc in msg.tool_calls
    ]
}) # sí hay tool calls --> ejecuta y devuelve rta

    for call in msg.tool_calls:
        tool_name = call.function.name
        tool_args = json.loads(call.function.arguments)

        print(f"\nAgente quiere utilizar: {tool_name} {" ".join(tool_args.values())}")
        result = execute_tool(tool_name, tool_args, supervision)

        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": str(result) if result is not None else "Error: la tool no devolvió resultado"
        })


def plan_mode_flow(user_message: str, messages: list) -> bool: # armo plan y espero confirmación del usuario
  print("\n[PLAN] Generando plan...")
  plan_messages = messages + [{
      "role": "user",
      "content": (
          f"Tarea: {user_message}\n\n"
          "Antes de hacer cualquier acción, describí detalladamente el plan de pasos "
          "que seguirías para completar esta tarea. No ejecutes ninguna tool todavía, "
          "solo listá los pasos."
      ) # agrego prompt al historial
  }]

  plan_response = client.chat.completions.create(
      model=MODEL,
      messages=plan_messages,
  )
  plan = plan_response.choices[0].message.content
  print(f"Plan propuesto:\n{plan}")

  choice = input("\n¿Aprobás este plan? (s/n/modificar): ").strip().lower()
  if choice == 'n':
      print("Tarea cancelada.")
      return False
  elif choice == 'modificar':
      modification = input("Describí los cambios al plan: ").strip() # prompt de modificación
      messages[-1]["content"] += f"\n\nModificación al plan: {modification}"
  return True


def run_agent(): # loop externo, chat interactua con agente. COmandos: plan, supervision, reset, exit
  messages = [{"role": "system", "content": PROMPT}]
  plan_mode = True # prendido opor defecto
  supervision = True

  print("Agente listo")
  print("="*50)
  print(f"Comandos:\n/plan (des/activa el paso a paso) | \n/supervision (des/activa control sobre operaciones críticas) | \n/reset (borra el historial) | \n/exit (abandonar chat) |")
  print(f"Estado inicial → Plan mode: {'ON' if plan_mode else 'OFF'} | Supervisión: {'ON' if supervision else 'OFF'}")

  while True: # loop ext espera input de user
      try:
          user_input = input("Prompt: ").strip()
      except (KeyboardInterrupt, EOFError):
          print("\nError. Saliendo...")
          break

      if not user_input:
          continue

      # Comandos especiales
      if user_input == "/exit":
          print("¡Hasta luego!")
          break
      elif user_input == "/reset":
          messages = [{"role": "system", "content": PROMPT}]
          print("Historial reseteado.")
          continue
      elif user_input == "/plan":
          plan_mode = not plan_mode
          print(f"Plan mode: {'ON' if plan_mode else 'OFF'}")
          continue
      elif user_input == "/supervision":
          supervision = not supervision
          print(f"Supervisión: {'ON' if supervision else 'OFF'}")
          continue

      messages.append({"role": "user", "content": user_input}) # msg de user al hisotiral

      # muestro plan --> pido aprobación
      if plan_mode:
          approved = plan_mode_flow(user_input, messages[:-1])
          if not approved:
              messages.pop()  # saco el mensaje del usuario si se canceló
              continue

      print("\nAgente: ", end="", flush=True) # loop inst ejecuta tools hasta rta final
      try:
          response = inner_loop(messages, supervision)
          print(f"\nAgente: {response}\n")
      except Exception as e:
          print(f"\nError en el agente: {e}\n")

## Ejecución

In [ ]:
REPO_URL = "https://github.com/facebook/create-react-app"
state = run_main_agent(repo_url=REPO_URL, supervision=False)

✓ Repo ya clonado en /content/workspace/create-react-app (memoria)
[MAIN] Tarea iniciada: Analizar el repositorio React en /content/workspace/create-react-app y generar reporte de arquitectura.
[EXPLORER] iniciando
  [EXPLORER] iteración 1
[EXPLORER] tool: list_files ['/content/workspace/create-react-app']
  [EXPLORER] iteración 2
[EXPLORER] tool: list_files ['/content/workspace/create-react-app/packages']
  [EXPLORER] iteración 3
[EXPLORER] tool: list_files ['/content/workspace/create-react-app/packages/create-react-app']
  [EXPLORER] iteración 4
[EXPLORER] tool: read_file ['/content/workspace/create-react-app/package.json']
  [EXPLORER] iteración 5
[EXPLORER] tool: read_file ['/content/workspace/create-react-app/README.md']
  [EXPLORER] iteración 6
[EXPLORER] tool: list_files ['/content/workspace/create-react-app/src']
  [EXPLORER] iteración 7
[EXPLORER] tool: list_files ['/content/workspace/create-react-app/packages/react-scripts']
  [EXPLORER] iteración 8
[EXPLORER] tool: list_file